In [0]:
# Resumo das tabelas criadas na camada Gold
print("="*80)
print("CAMADA GOLD - DATASETS CRIADOS COM SUCESSO")
print("="*80)

tabelas_gold = [
    "indicadores_municipio",
    "metas_vs_resultados_municipio",
    "metas_vs_resultados_uf",
    "evolucao_temporal_municipio",
    "agregacoes_uf",
    "metricas_brasil",
    "features_ml"
]

for tabela in tabelas_gold:
    count = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{tabela}").count()
    colunas = len(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{tabela}").columns)
    print(f"✓ {GOLD_CATALOG}.{GOLD_SCHEMA}.{tabela}")
    print(f"  - Registros: {count:,}")
    print(f"  - Colunas: {colunas}")
    print()

print("="*80)
print("DATASETS PRONTOS PARA:")
print("  ✓ Dashboards e visualizações")
print("  ✓ Análises estatísticas")
print("  ✓ Treinamento de modelos de Machine Learning")
print("="*80)

CAMADA GOLD - DATASETS CRIADOS COM SUCESSO
✓ workspace.gold.indicadores_municipio
  - Registros: 23,995
  - Colunas: 18

✓ workspace.gold.metas_vs_resultados_municipio
  - Registros: 74,928
  - Colunas: 10

✓ workspace.gold.metas_vs_resultados_uf
  - Registros: 378
  - Colunas: 9

✓ workspace.gold.evolucao_temporal_municipio
  - Registros: 23,995
  - Colunas: 11

✓ workspace.gold.agregacoes_uf
  - Registros: 145
  - Colunas: 11

✓ workspace.gold.metricas_brasil
  - Registros: 7
  - Colunas: 17

✓ workspace.gold.features_ml
  - Registros: 23,995
  - Colunas: 23

DATASETS PRONTOS PARA:
  ✓ Dashboards e visualizações
  ✓ Análises estatísticas
  ✓ Treinamento de modelos de Machine Learning


In [0]:
# Exemplo de análise: Top 10 UFs por taxa de alfabetização (2023)
top_ufs = (
    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.agregacoes_uf")
    .filter((F.col("ano") == 2023) & (F.col("rede") == "publica"))
    .orderBy(F.desc("taxa_alfabetizacao_media"))
    .select(
        "ranking_nacional",
        "sigla_uf",
        "taxa_alfabetizacao_media",
        "classificacao"
    )
    .limit(10)
)

print("\n🏆 TOP 10 ESTADOS - ALFABETIZAÇÃO 2023 (Rede Pública)")
print("="*70)
display(top_ufs)


🏆 TOP 10 ESTADOS - ALFABETIZAÇÃO 2023 (Rede Pública)


ranking_nacional,sigla_uf,taxa_alfabetizacao_media,classificacao
1,CE,84.46,Top 5
2,PR,73.12,Top 5
3,ES,67.93,Top 5
4,GO,66.74,Top 5
5,RO,64.6,Top 5
6,RS,63.43,Top 10
7,SC,61.38,Top 10
8,MG,59.81,Top 10
9,PE,58.95,Top 10
10,MA,56.43,Top 10


In [0]:
# Exemplo de análise: Evolução temporal nacional
evolucao_nacional = (
    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.metricas_brasil")
    .filter(F.col("rede") == "publica")
    .orderBy("ano")
    .select(
        "ano",
        "taxa_alfabetizacao_media_brasil",
        "media_portugues_brasil",
        "num_municipios_unicos",
        "meta_2024",
        "meta_2030"
    )
)

print("\n📈 EVOLUÇÃO TEMPORAL - BRASIL (Rede Pública)")
print("="*70)
display(evolucao_nacional)


📈 EVOLUÇÃO TEMPORAL - BRASIL (Rede Pública)


ano,taxa_alfabetizacao_media_brasil,media_portugues_brasil,num_municipios_unicos,meta_2024,meta_2030
2023,60.46918181818192,750.3406956363639,4950,59.9,80.0
2024,63.16514321972449,753.2314266316147,5516,59.9,80.0


In [0]:
# Exemplo de análise estatística: Distribuição de status de metas
distribuicao_metas = (
    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.metas_vs_resultados_uf")
    .filter((F.col("ano") == 2023) & (F.col("ano_meta") == 2024))
    .groupBy("status_meta")
    .agg(
        F.count("*").alias("num_ufs"),
        F.avg("percentual_alcance").alias("percentual_alcance_medio"),
        F.avg("gap_meta").alias("gap_medio")
    )
    .orderBy(F.desc("num_ufs"))
)

print("\n🎯 DISTRIBUIÇÃO DE ATINGIMENTO DE METAS 2024")
print("="*70)
display(distribuicao_metas)


🎯 DISTRIBUIÇÃO DE ATINGIMENTO DE METAS 2024


status_meta,num_ufs,percentual_alcance_medio,gap_medio
Próximo,16,94.0556036327645,3.5518750000000003
Em Progresso,7,86.47571639174829,6.164285714285713
Abaixo do Esperado,3,null,null
Atingida,1,105.62499999999999,-4.5


In [0]:
# Dataset 5: Agregações Estaduais
# Consolida indicadores por UF com rankings e comparações

agregacoes_uf = (
    indicador_uf
    .groupBy("ano", "sigla_uf", "rede")
    .agg(
        F.avg("taxa_alfabetizacao").alias("taxa_alfabetizacao_media"),
        F.max("taxa_alfabetizacao").alias("taxa_alfabetizacao_max"),
        F.min("taxa_alfabetizacao").alias("taxa_alfabetizacao_min"),
        F.stddev("taxa_alfabetizacao").alias("taxa_alfabetizacao_desvio"),
        F.avg("media_portugues").alias("media_portugues_uf"),
        F.count("*").alias("num_registros")
    )
)

# Adicionar ranking nacional
window_ranking = Window.partitionBy("ano", "rede").orderBy(F.desc("taxa_alfabetizacao_media"))

agregacoes_uf = (
    agregacoes_uf
    .withColumn("ranking_nacional", F.row_number().over(window_ranking))
    .withColumn(
        "classificacao",
        F.when(F.col("ranking_nacional") <= 5, "Top 5")
        .when(F.col("ranking_nacional") <= 10, "Top 10")
        .when(F.col("ranking_nacional") >= 22, "Bottom 5")
        .otherwise("Intermediário")
    )
)

# Salvar tabela
agregacoes_uf.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.agregacoes_uf"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.agregacoes_uf criada")
print(f"  Total de registros: {agregacoes_uf.count():,}")
display(agregacoes_uf.filter(F.col("ano") == 2023).orderBy("ranking_nacional").limit(10))

✓ Tabela workspace.gold.agregacoes_uf criada
  Total de registros: 145


ano,sigla_uf,rede,taxa_alfabetizacao_media,taxa_alfabetizacao_max,taxa_alfabetizacao_min,taxa_alfabetizacao_desvio,media_portugues_uf,num_registros,ranking_nacional,classificacao
2023,PR,estadual,80.29,80.29,80.29,null,770.8324,1,1,Top 5
2023,CE,municipal,84.49,84.49,84.49,null,795.7293,1,1,Top 5
2023,CE,publica,84.46,84.46,84.46,null,795.675,1,1,Top 5
2023,PE,estadual,79.25,79.25,79.25,null,763.6886,1,2,Top 5
2023,PR,municipal,73.11,73.11,73.11,null,757.1952,1,2,Top 5
2023,PR,publica,73.12,73.12,73.12,null,757.2146,1,2,Top 5
2023,CE,estadual,78.75,78.75,78.75,null,785.1658,1,3,Top 5
2023,ES,municipal,67.52,67.52,67.52,null,753.1398,1,3,Top 5
2023,ES,publica,67.93,67.93,67.93,null,753.7213,1,3,Top 5
2023,GO,municipal,66.72,66.72,66.72,null,752.1315,1,4,Top 5


In [0]:
# Dataset 6: Métricas Nacionais Consolidadas
# KPIs agregados para visão executiva e dashboards

# Agregar indicadores de todos os municípios para visão Brasil
metricas_brasil = (
    indicador_municipio
    .groupBy("ano", "rede")
    .agg(
        F.avg("taxa_alfabetizacao").alias("taxa_alfabetizacao_media_brasil"),
        F.max("taxa_alfabetizacao").alias("taxa_alfabetizacao_max_brasil"),
        F.min("taxa_alfabetizacao").alias("taxa_alfabetizacao_min_brasil"),
        F.stddev("taxa_alfabetizacao").alias("taxa_alfabetizacao_desvio_brasil"),
        F.avg("media_portugues").alias("media_portugues_brasil"),
        F.count("id_municipio").alias("num_municipios"),
        F.countDistinct("id_municipio").alias("num_municipios_unicos")
    )
)

# Adicionar metas nacionais
metricas_com_metas = (
    metricas_brasil
    .join(
        meta_brasil.select(
            "ano",
            "rede",
            "taxa_alfabetizacao",
            "meta_2024",
            "meta_2025",
            "meta_2026",
            "meta_2027",
            "meta_2028",
            "meta_2029",
            "meta_2030"
        ),
        on=["ano", "rede"],
        how="left"
    )
    .withColumnRenamed("taxa_alfabetizacao", "taxa_oficial_brasil")
)

# Salvar tabela
metricas_com_metas.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.metricas_brasil"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.metricas_brasil criada")
print(f"  Total de registros: {metricas_com_metas.count():,}")
display(metricas_com_metas.orderBy(F.desc("ano")))

✓ Tabela workspace.gold.metricas_brasil criada
  Total de registros: 7


ano,rede,taxa_alfabetizacao_media_brasil,taxa_alfabetizacao_max_brasil,taxa_alfabetizacao_min_brasil,taxa_alfabetizacao_desvio_brasil,media_portugues_brasil,num_municipios,num_municipios_unicos,taxa_oficial_brasil,meta_2024,meta_2025,meta_2026,meta_2027,meta_2028,meta_2029,meta_2030
2024,municipal,62.79976688693109,100.0,4.4,19.34214809319735,752.8517387481638,5448,5448,null,null,null,null,null,null,null,null
2024,total,36.49203517587939,75.47,12.5,10.553539100002586,726.5136432160798,398,398,null,null,null,null,null,null,null,null
2024,estadual,62.90695211786369,100.0,4.76,19.13809811977002,752.1896952117863,1086,1086,null,null,null,null,null,null,null,null
2024,publica,63.16514321972449,100.0,4.4,18.99925748029889,753.2314266316147,5516,5516,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0
2023,municipal,60.27846549192365,100.0,4.35,19.822220079506582,750.4037856828201,5448,5448,null,null,null,null,null,null,null,null
2023,publica,60.46918181818192,100.0,4.56,20.00463617605536,750.3406956363639,4950,4950,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0
2023,estadual,63.75064403829416,100.0,2.12,20.17718625218185,752.378226283724,1149,1149,null,null,null,null,null,null,null,null


In [0]:
# Dataset 7: Features para Machine Learning
# Dataset enriquecido com features prontas para modelagem

# Criar features derivadas e agregações para ML
features_ml = (
    indicador_municipio
    .alias("ind")
    .join(
        meta_municipio.select(
            "ano",
            "id_municipio",
            "rede",
            "meta_2030",
            "percentual_participacao"
        ).alias("meta"),
        on=["ano", "id_municipio", "rede"],
        how="left"
    )
    .withColumn("codigo_uf", F.substring(F.col("ind.id_municipio"), 1, 2))
    .withColumn(
        "gap_meta_2030",
        F.col("meta.meta_2030") - F.col("ind.taxa_alfabetizacao")
    )
    .withColumn(
        "soma_niveis_basicos",
        F.col("ind.proporcao_aluno_nivel_0") + 
        F.col("ind.proporcao_aluno_nivel_1") + 
        F.col("ind.proporcao_aluno_nivel_2")
    )
    .withColumn(
        "soma_niveis_avancados",
        F.col("ind.proporcao_aluno_nivel_6") + 
        F.col("ind.proporcao_aluno_nivel_7") + 
        F.col("ind.proporcao_aluno_nivel_8")
    )
    .withColumn(
        "indice_concentracao",
        F.greatest(
            F.col("ind.proporcao_aluno_nivel_0"),
            F.col("ind.proporcao_aluno_nivel_1"),
            F.col("ind.proporcao_aluno_nivel_2"),
            F.col("ind.proporcao_aluno_nivel_3"),
            F.col("ind.proporcao_aluno_nivel_4"),
            F.col("ind.proporcao_aluno_nivel_5"),
            F.col("ind.proporcao_aluno_nivel_6"),
            F.col("ind.proporcao_aluno_nivel_7"),
            F.col("ind.proporcao_aluno_nivel_8")
        )
    )
    .withColumn(
        "rede_numerica",
        F.when(F.col("ind.rede") == "privada", 1)
        .when(F.col("ind.rede") == "municipal", 2)
        .when(F.col("ind.rede") == "estadual", 3)
        .when(F.col("ind.rede") == "federal", 4)
        .otherwise(5)
    )
    .select(
        F.col("ind.ano"),
        F.col("ind.id_municipio"),
        F.col("codigo_uf"),
        F.col("ind.serie"),
        F.col("ind.rede"),
        F.col("rede_numerica"),
        F.col("ind.taxa_alfabetizacao"),
        F.col("ind.media_portugues"),
        F.col("meta.meta_2030"),
        F.col("gap_meta_2030"),
        F.col("meta.percentual_participacao"),
        F.col("soma_niveis_basicos"),
        F.col("soma_niveis_avancados"),
        F.col("indice_concentracao"),
        F.col("ind.proporcao_aluno_nivel_0"),
        F.col("ind.proporcao_aluno_nivel_1"),
        F.col("ind.proporcao_aluno_nivel_2"),
        F.col("ind.proporcao_aluno_nivel_3"),
        F.col("ind.proporcao_aluno_nivel_4"),
        F.col("ind.proporcao_aluno_nivel_5"),
        F.col("ind.proporcao_aluno_nivel_6"),
        F.col("ind.proporcao_aluno_nivel_7"),
        F.col("ind.proporcao_aluno_nivel_8")
    )
)

# Salvar tabela
features_ml.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.features_ml"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.features_ml criada")
print(f"  Total de registros: {features_ml.count():,}")
print(f"  Features disponíveis: {len(features_ml.columns)}")
display(features_ml.limit(5))

✓ Tabela workspace.gold.features_ml criada
  Total de registros: 23,995
  Features disponíveis: 23


ano,id_municipio,codigo_uf,serie,rede,rede_numerica,taxa_alfabetizacao,media_portugues,meta_2030,gap_meta_2030,percentual_participacao,soma_niveis_basicos,soma_niveis_avancados,indice_concentracao,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
2023,1100031,11,2,municipal,2,69.1,767.8763,80.0,10.900000000000006,90.48,null,null,null,null,null,null,null,null,null,null,null,null
2023,1100072,11,2,municipal,2,58.2,747.8918,80.0,21.799999999999997,92.25,null,null,null,null,null,null,null,null,null,null,null,null
2023,1100189,11,2,publica,5,69.73,762.4062,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2023,1101609,11,2,municipal,2,50.7,745.6802,80.0,29.299999999999997,95.0,null,null,null,null,null,null,null,null,null,null,null,null
2023,1101807,11,2,municipal,2,55.69,752.3724,80.0,24.310000000000002,83.15,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
# Dataset 2: Comparação entre Metas e Resultados por Município
# Análise de atingimento de metas por ano e rede

# Unpivot das metas para facilitar análise temporal
metas_unpivot = (
    meta_municipio
    .select(
        "ano",
        "id_municipio",
        "rede",
        "taxa_alfabetizacao",
        F.expr("stack(7, "
               "'2024', meta_2024, "
               "'2025', meta_2025, "
               "'2026', meta_2026, "
               "'2027', meta_2027, "
               "'2028', meta_2028, "
               "'2029', meta_2029, "
               "'2030', meta_2030) as (ano_meta, meta)")
    )
    .withColumn("ano_meta", F.col("ano_meta").cast("int"))
)

# Comparação: pegar o resultado do ano base e comparar com as metas
metas_vs_resultados_municipio = (
    metas_unpivot
    .withColumn("codigo_uf", F.substring(F.col("id_municipio"), 1, 2))
    .withColumn(
        "gap_meta",
        F.col("meta") - F.col("taxa_alfabetizacao")
    )
    .withColumn(
        "percentual_alcance",
        (F.col("taxa_alfabetizacao") / F.col("meta")) * 100
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_alfabetizacao") >= F.col("meta"), "Atingida")
        .when(F.col("percentual_alcance") >= 90, "Próximo")
        .when(F.col("percentual_alcance") >= 75, "Em Progresso")
        .otherwise("Abaixo do Esperado")
    )
    .select(
        "ano",
        "ano_meta",
        "id_municipio",
        "codigo_uf",
        "rede",
        "taxa_alfabetizacao",
        "meta",
        "gap_meta",
        "percentual_alcance",
        "status_meta"
    )
)

# Salvar tabela
metas_vs_resultados_municipio.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.metas_vs_resultados_municipio"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.metas_vs_resultados_municipio criada")
print(f"  Total de registros: {metas_vs_resultados_municipio.count():,}")
display(metas_vs_resultados_municipio.limit(5))

✓ Tabela workspace.gold.metas_vs_resultados_municipio criada
  Total de registros: 74,928


ano,ano_meta,id_municipio,codigo_uf,rede,taxa_alfabetizacao,meta,gap_meta,percentual_alcance,status_meta
2023,2024,4301750,43,municipal,null,null,null,null,Abaixo do Esperado
2024,2024,4301750,43,municipal,4.4,null,null,null,Abaixo do Esperado
2024,2024,2406908,24,municipal,42.86,7.94,-34.92,539.7984886649874,Atingida
2023,2024,2406908,24,municipal,4.4,7.94,3.54,55.41561712846348,Abaixo do Esperado
2023,2024,1718501,17,municipal,4.6,8.25,3.6500000000000004,55.75757575757575,Abaixo do Esperado


In [0]:
# Dataset 3: Comparação entre Metas e Resultados por UF

# Unpivot das metas de UF
metas_uf_unpivot = (
    meta_uf
    .select(
        "ano",
        "sigla_uf",
        "rede",
        "taxa_alfabetizacao",
        F.expr("stack(7, "
               "'2024', meta_2024, "
               "'2025', meta_2025, "
               "'2026', meta_2026, "
               "'2027', meta_2027, "
               "'2028', meta_2028, "
               "'2029', meta_2029, "
               "'2030', meta_2030) as (ano_meta, meta)")
    )
    .withColumn("ano_meta", F.col("ano_meta").cast("int"))
)

# Comparação metas vs resultados
metas_vs_resultados_uf = (
    metas_uf_unpivot
    .withColumn(
        "gap_meta",
        F.col("meta") - F.col("taxa_alfabetizacao")
    )
    .withColumn(
        "percentual_alcance",
        (F.col("taxa_alfabetizacao") / F.col("meta")) * 100
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_alfabetizacao") >= F.col("meta"), "Atingida")
        .when(F.col("percentual_alcance") >= 90, "Próximo")
        .when(F.col("percentual_alcance") >= 75, "Em Progresso")
        .otherwise("Abaixo do Esperado")
    )
    .select(
        "ano",
        "ano_meta",
        "sigla_uf",
        "rede",
        "taxa_alfabetizacao",
        "meta",
        "gap_meta",
        "percentual_alcance",
        "status_meta"
    )
)

# Salvar tabela
metas_vs_resultados_uf.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.metas_vs_resultados_uf"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.metas_vs_resultados_uf criada")
print(f"  Total de registros: {metas_vs_resultados_uf.count():,}")
display(metas_vs_resultados_uf.limit(5))

✓ Tabela workspace.gold.metas_vs_resultados_uf criada
  Total de registros: 378


ano,ano_meta,sigla_uf,rede,taxa_alfabetizacao,meta,gap_meta,percentual_alcance,status_meta
2024,2024,RR,publica,null,null,null,null,Abaixo do Esperado
2023,2024,RR,publica,null,null,null,null,Abaixo do Esperado
2023,2024,AC,publica,null,null,null,null,Abaixo do Esperado
2024,2024,AC,publica,51.38,null,null,null,Abaixo do Esperado
2023,2024,DF,publica,null,null,null,null,Abaixo do Esperado


In [0]:
# Dataset 4: Evolução Temporal dos Indicadores por Município
# Séries históricas com cálculo de variação ano a ano

# Window para calcular variações
window_municipio = Window.partitionBy("id_municipio", "rede").orderBy("ano")

evolucao_temporal_municipio = (
    indicador_municipio
    .withColumn(
        "taxa_ano_anterior",
        F.lag("taxa_alfabetizacao").over(window_municipio)
    )
    .withColumn(
        "variacao_absoluta",
        F.col("taxa_alfabetizacao") - F.col("taxa_ano_anterior")
    )
    .withColumn(
        "variacao_percentual",
        F.when(
            F.col("taxa_ano_anterior").isNotNull() & (F.col("taxa_ano_anterior") != 0),
            ((F.col("taxa_alfabetizacao") - F.col("taxa_ano_anterior")) / F.col("taxa_ano_anterior")) * 100
        ).otherwise(None)
    )
    .withColumn(
        "tendencia",
        F.when(F.col("variacao_absoluta") > 2, "Crescimento Forte")
        .when(F.col("variacao_absoluta") > 0, "Crescimento")
        .when(F.col("variacao_absoluta") < -2, "Queda Forte")
        .when(F.col("variacao_absoluta") < 0, "Queda")
        .otherwise("Estável")
    )
    .withColumn("codigo_uf", F.substring(F.col("id_municipio"), 1, 2))
    .select(
        "ano",
        "id_municipio",
        "codigo_uf",
        "serie",
        "rede",
        "taxa_alfabetizacao",
        "taxa_ano_anterior",
        "variacao_absoluta",
        "variacao_percentual",
        "tendencia",
        "media_portugues"
    )
)

# Salvar tabela
evolucao_temporal_municipio.write.mode("overwrite").saveAsTable(
    f"{GOLD_CATALOG}.{GOLD_SCHEMA}.evolucao_temporal_municipio"
)

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.evolucao_temporal_municipio criada")
print(f"  Total de registros: {evolucao_temporal_municipio.count():,}")
display(evolucao_temporal_municipio.filter(F.col("ano") == 2023).limit(5))

✓ Tabela workspace.gold.evolucao_temporal_municipio criada
  Total de registros: 23,995


ano,id_municipio,codigo_uf,serie,rede,taxa_alfabetizacao,taxa_ano_anterior,variacao_absoluta,variacao_percentual,tendencia,media_portugues
2023,1100015,11,2,municipal,64.55,null,null,null,Estável,758.3304
2023,1100015,11,2,publica,64.55,null,null,null,Estável,758.3304
2023,1100023,11,2,municipal,62.3,null,null,null,Estável,757.0999
2023,1100023,11,2,publica,62.3,null,null,null,Estável,757.0999
2023,1100031,11,2,municipal,69.1,null,null,null,Estável,767.8763


# Gold Layer - Camada Analítica

Este notebook cria datasets prontos para análise, consolidando informações das camadas Bronze e Silver.

## Datasets Criados:

### 1. **Indicadores por Município** (`gold.indicadores_municipio`)
- Indicadores de alfabetização consolidados por município
- Informações demográficas e geográficas
- Métricas de proficiência e distribuição por níveis

### 2. **Comparação Metas vs Resultados** (`gold.metas_vs_resultados`)
- Análise de atingimento de metas por região
- Gaps e percentuais de alcance
- Tendências de desempenho

### 3. **Evolução Temporal** (`gold.evolucao_temporal`)
- Séries históricas de indicadores
- Taxas de crescimento ano a ano
- Análise de tendências

### 4. **Agregações Estaduais** (`gold.agregacoes_uf`)
- Indicadores consolidados por UF
- Rankings e comparações regionais

### 5. **Métricas Nacionais** (`gold.metricas_brasil`)
- KPIs nacionais consolidados
- Visão macro para dashboards executivos

**Arquitetura Unity Catalog:**
- **Entrada**: `workspace.silver.*`
- **Saída**: `workspace.gold.*`

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("tech-challenge-gold").getOrCreate()

# Configuração Unity Catalog
SILVER_CATALOG = "workspace"
SILVER_SCHEMA = "silver"
GOLD_CATALOG = "workspace"
GOLD_SCHEMA = "gold"

print(f"Entrada: {SILVER_CATALOG}.{SILVER_SCHEMA}")
print(f"Saída: {GOLD_CATALOG}.{GOLD_SCHEMA}")

# Criar schema gold se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")
print(f"✓ Schema {GOLD_CATALOG}.{GOLD_SCHEMA} pronto")

Entrada: workspace.silver
Saída: workspace.gold
✓ Schema workspace.gold pronto


In [0]:
# Carregar tabelas da camada Silver
indicador_municipio = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.indicador_alfabetizacao_municipio")
indicador_uf = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.indicador_alfabetizacao_uf")
meta_municipio = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.meta_alfabetizacao_municipio")
meta_uf = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.meta_alfabetizacao_uf")
meta_brasil = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.meta_alfabetizacao_brasil")
microdados = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.microdados_alunos")

print("✓ Tabelas Silver carregadas com sucesso")

✓ Tabelas Silver carregadas com sucesso


In [0]:
# Dataset 1: Indicadores de Alfabetização por Município
# Consolida informações de indicadores com metadados geográficos

indicadores_municipio = (
    indicador_municipio
    .withColumn(
        "codigo_uf",
        F.substring(F.col("id_municipio"), 1, 2)
    )
    .withColumn(
        "nivel_alfabetizacao",
        F.when(F.col("taxa_alfabetizacao") >= 80, "Alto")
        .when(F.col("taxa_alfabetizacao") >= 60, "Médio")
        .when(F.col("taxa_alfabetizacao") >= 40, "Baixo")
        .otherwise("Muito Baixo")
    )
    .withColumn(
        "faixa_proficiencia",
        F.when(F.col("media_portugues") >= 800, "Avançado")
        .when(F.col("media_portugues") >= 750, "Adequado")
        .when(F.col("media_portugues") >= 700, "Básico")
        .otherwise("Abaixo do Básico")
    )
    .select(
        "ano",
        "id_municipio",
        "codigo_uf",
        "serie",
        "rede",
        "taxa_alfabetizacao",
        "nivel_alfabetizacao",
        "media_portugues",
        "faixa_proficiencia",
        "proporcao_aluno_nivel_0",
        "proporcao_aluno_nivel_1",
        "proporcao_aluno_nivel_2",
        "proporcao_aluno_nivel_3",
        "proporcao_aluno_nivel_4",
        "proporcao_aluno_nivel_5",
        "proporcao_aluno_nivel_6",
        "proporcao_aluno_nivel_7",
        "proporcao_aluno_nivel_8"
    )
)

# Salvar como tabela Delta no Unity Catalog
indicadores_municipio.write.mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.indicadores_municipio")

print(f"✓ Tabela {GOLD_CATALOG}.{GOLD_SCHEMA}.indicadores_municipio criada")
print(f"  Total de registros: {indicadores_municipio.count():,}")
display(indicadores_municipio.limit(5))

✓ Tabela workspace.gold.indicadores_municipio criada
  Total de registros: 23,995


ano,id_municipio,codigo_uf,serie,rede,taxa_alfabetizacao,nivel_alfabetizacao,media_portugues,faixa_proficiencia,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
2023,1100031,11,2,municipal,69.1,Médio,767.8763,Adequado,null,null,null,null,null,null,null,null,null
2023,1100072,11,2,municipal,58.2,Baixo,747.8918,Básico,null,null,null,null,null,null,null,null,null
2023,1100189,11,2,publica,69.73,Médio,762.4062,Adequado,null,null,null,null,null,null,null,null,null
2023,1101609,11,2,municipal,50.7,Baixo,745.6802,Básico,null,null,null,null,null,null,null,null,null
2023,1101807,11,2,municipal,55.69,Baixo,752.3724,Adequado,null,null,null,null,null,null,null,null,null
